# US-05 — Validation d'une SparkSession distribuée

**EPIC-01 — Infrastructure** · Responsable : Zakaria

Ce notebook valide les critères suivants :

| Ticket | Critère |
|---|---|
| SBD-69 | Une SparkSession se connecte à `spark://spark-master:7077` |
| SBD-70 | La version de Spark, le Master, l'appName et `sc.defaultParallelism` sont affichés |
| SBD-71 | Un traitement Spark simple s'exécute sur le cluster |
| SBD-72 | Une capture de Spark Master UI montre les deux Workers actifs |

Le topic Kafka `transactions` (3 partitions) et le bucket MinIO `lake` ont été créés en ligne de commande (voir `docs/epics/EPIC-01-infrastructure.md`).

## 1. Connexion au cluster (SBD-69)

Le Driver s'exécute dans le conteneur Jupyter. Il contacte le Master, qui lance des Executors sur les deux Workers.
`spark.driver.host = jupyter` permet aux Workers de joindre le Driver par le réseau Docker.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SahlPay-US05-Validation-Cluster")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.cores", "2")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .getOrCreate()
)

sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/07 13:29:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Informations sur la session (SBD-70)

`defaultParallelism` attendu : **4** (2 Workers × 2 cœurs).

In [2]:
print("Spark version      :", spark.version)
print("Spark Master       :", sc.master)
print("Application        :", sc.appName)
print("Application ID     :", sc.applicationId)
print("defaultParallelism :", sc.defaultParallelism)

Spark version      : 3.5.7
Spark Master       : spark://spark-master:7077
Application        : SahlPay-US05-Validation-Cluster
Application ID     : app-20261107132927-0000
defaultParallelism : 4


## 3. Traitement distribué (SBD-71)

Somme des carrés de 1 à 1 000 000, répartie sur 4 partitions.
- `parallelize` et `map` sont des **transformations** (paresseuses).
- `sum` est une **action** : elle déclenche l'exécution sur les Executors.

Le résultat attendu est n(n+1)(2n+1)/6 = 333 333 833 333 500 000.

In [3]:
import time

n = 1_000_000
rdd = sc.parallelize(range(1, n + 1), numSlices=sc.defaultParallelism)

start = time.time()
total = rdd.map(lambda x: x * x).sum()
duration = time.time() - start

expected = n * (n + 1) * (2 * n + 1) // 6

print("Partitions       :", rdd.getNumPartitions())
print("Somme des carrés :", total)
print("Résultat attendu :", expected)
print("Résultat correct :", total == expected)
print(f"Durée            : {duration:.2f} s")

[Stage 0:>                                                          (0 + 4) / 4]

Partitions       : 4
Somme des carrés : 333333833333500000
Résultat attendu : 333333833333500000
Résultat correct : True
Durée            : 1.26 s


Vérification que les tâches ont été exécutées par des Executors distants (et non par le Driver) :

In [4]:
import socket

hosts = (
    sc.parallelize(range(8), 8)
    .map(lambda _: socket.gethostname())
    .distinct()
    .collect()
)

print("Executors ayant exécuté les tâches :", hosts)
print("Nombre d'Executors                 :", len(hosts))

[Stage 1:==============>                                            (2 + 4) / 8]

Executors ayant exécuté les tâches : ['spark-worker-2', 'spark-worker-1']
Nombre d'Executors                 : 2


## 4. Capture de Spark Master UI (SBD-72)

**Avant d'exécuter la cellule suivante**, ouvrir http://localhost:8080 et réaliser la capture :
- *Alive Workers : 2* et les deux Workers à l'état **ALIVE** ;
- l'application `SahlPay-US05-Validation-Cluster` dans *Running Applications*.

Fichier : `docs/screenshots/EPIC-01-US-05_sparkMaster_2Workers_application.png`

## 5. Libération des ressources

In [5]:
spark.stop()